# BraTS 2021 nnU-Net (DynUNet) 3D Segmentation Baseline

This notebook implements an end-to-end 3D multi-modal brain tumor segmentation baseline on **BraTS 2021** using **MONAI's DynUNet** (the official, reputable implementation of nnU-Net's architecture) and **PyTorch**.

### nnU-Net Architectural Features
- **Network**: `monai.networks.nets.DynUNet` with multi-stage isotropic/anisotropic kernel configuration.
- **Normalization**: **Instance Normalization** (`norm_name='instance'`), which prevents inter-sample batch statistics interference in 3D medical volumes.
- **Activation**: **LeakyReLU** (`negative_slope=0.01`, inplace=True).
- **Loss**: `DiceCELoss(sigmoid=True)` (Dice + Binary Cross-Entropy combination).
- **Optimizer**: `AdamW` (or `SGD` with Nesterov momentum = 0.99) + Cosine Annealing LR Scheduler.
- **Target Sub-regions**:
  - **Whole Tumor (WT)**: Labels 1 (NCR) + 2 (ED) + 4 (ET)
  - **Tumor Core (TC)**: Labels 1 (NCR) + 4 (ET)
  - **Enhancing Tumor (ET)**: Label 4 (ET)
- **Adaptive Execution**: Runs seamlessly on **Google Colab (GPU)**, **Windows (CUDA/CPU)**, and **macOS (Apple Silicon MPS/CPU)** with zero hardcoded paths.

## 1. Environment Detection and Branching Setup

Automatically detects **Google Colab** vs **Local Environment**, configures GPU acceleration (NVIDIA CUDA or Apple Silicon MPS), and handles Colab package installation and Google Drive mounting.

In [ ]:
import os
import sys
from pathlib import Path

# 1. Auto-detect environment
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Environment detected: {'Google Colab' if IS_COLAB else 'Local Environment'}")

# 2. Branching setup for Google Colab
if IS_COLAB:
    print("Running Colab initialization...")
    !nvidia-smi
    !pip install -q monai nibabel matplotlib scikit-learn tqdm
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive"):
            drive.mount("/content/drive")
            print("Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Drive mount status: {e}")
else:
    print("Running in local environment; skipping Colab package installation and Drive mount.")

## 2. Dynamic Project Root and Imports

Locate project root dynamically without hardcoded paths, add to `sys.path`, and import `src` modules.

In [ ]:
# Locate project root dynamically
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "config.py").is_file():
            return base.resolve()
    if IS_COLAB and Path("/content").exists():
        for candidate in Path("/content").rglob("src/config.py"):
            return candidate.parent.parent.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
print(f"Resolved Project Root: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import BaselineConfig, get_nnunet_config, resolve_dataset_dir, get_default_device
from src.eda import (
    scan_brats_dataset,
    compute_volume_statistics,
    analyze_label_distribution,
    plot_patient_modalities
)
from src.dataset import build_dataloaders, get_brats_data_dicts
from src.model import build_nnunet_model, build_model, get_model_summary
from src.loss_metric import build_loss_fn, BraTSMetric
from src.trainer import BraTSTrainer
from src.utils import set_seed, plot_training_curves, visualize_predictions, load_checkpoint

print("All modular components successfully imported.")

## 3. Dynamic Dataset Discovery and nnU-Net Configuration

The dataset path is resolved dynamically without hardcoded paths.
You can specify `CUSTOM_DATASET_PATH` to point to a custom folder if desired.

In [ ]:
# Optional custom path override: set to None for automatic discovery
# Example Colab custom path: '/content/drive/MyDrive/BraTS2021_Training_Data'
CUSTOM_DATASET_PATH = None

# Dynamically resolve dataset path
DATASET_DIR = resolve_dataset_dir(CUSTOM_DATASET_PATH)
print(f"Resolved Dataset Directory: {DATASET_DIR}")

# Initialize nnU-Net configuration
config = get_nnunet_config(data_dir=DATASET_DIR)
set_seed(config.seed)

# Create checkpoint and output directories dynamically
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.output_dir, exist_ok=True)

print(f"Execution Device: {config.device}")
print(f"Architecture: nnU-Net (DynUNet with {config.nnunet_norm} normalization)")
print(f"Stage Filters: {config.nnunet_filters}")
print(f"Checkpoint Directory: {os.path.abspath(config.checkpoint_dir)}")
print(f"Output Directory: {os.path.abspath(config.output_dir)}")

## 4. Exploratory Data Analysis (EDA)

Verify complete patient cases, inspect volume dimensions and voxel spacing, and visualize 4 MRI sequences and Ground Truth masks.

In [ ]:
# Scan all patient cases
patient_records = scan_brats_dataset(config.data_dir)
print(f"Total valid patient cases found: {len(patient_records)}")

if len(patient_records) == 0:
    raise FileNotFoundError(
        f"No patient cases found in '{config.data_dir}'. "
        "Please verify your dataset directory or set CUSTOM_DATASET_PATH in section 3."
    )

# Sample patient metadata and statistics
sample_patient = patient_records[0]
print(f"\nSample Patient ID: {sample_patient['patient_id']}")

label_stats = analyze_label_distribution(sample_patient['seg'])
print("\nSegmentation Label Breakdown:")
print(f"  Total Voxels: {label_stats['total_voxels']:,}")
print(f"  Background Voxels: {label_stats['background_voxels']:,} ({100*(1 - label_stats['wt_ratio']):.2f}%)")
print(f"  Whole Tumor (WT: 1+2+4): {label_stats['wt_voxels']:,} ({100*label_stats['wt_ratio']:.2f}%)")
print(f"  Tumor Core (TC: 1+4): {label_stats['tc_voxels']:,} ({100*label_stats['tc_ratio']:.2f}%)")
print(f"  Enhancing Tumor (ET: 4): {label_stats['et_voxels']:,} ({100*label_stats['et_ratio']:.2f}%)")

# Visualize multi-modal slices and target masks
eda_fig_path = os.path.join(config.output_dir, "eda_sample_patient.png")
fig = plot_patient_modalities(
    patient_record=sample_patient,
    slice_idx=None,
    save_path=eda_fig_path
)
plt.show()

## 5. Dataset and DataLoaders Setup

Builds training and validation DataLoaders with MONAI transformations:
- Training: Random 3D crops `(96, 96, 96)` and axis flips.
- Validation: Preserves full 3D volumes for sliding window inference.

In [ ]:
# Adaptive batch size based on available hardware
batch_size = config.batch_size if torch.cuda.is_available() else 1
num_workers = config.num_workers if torch.cuda.is_available() else 0

# Limit sample count for quick debugging if desired (None uses all cases)
MAX_SAMPLES = None

train_loader, val_loader, train_files, val_files = build_dataloaders(
    data_dir=config.data_dir,
    train_ratio=config.train_ratio,
    batch_size=batch_size,
    val_batch_size=config.val_batch_size,
    num_workers=num_workers,
    roi_size=config.roi_size,
    cache_rate=config.cache_rate,
    seed=config.seed,
    max_samples=MAX_SAMPLES
)

print(f"Training Cases: {len(train_files)} (Batches per epoch: {len(train_loader)})")
print(f"Validation Cases: {len(val_files)} (Batches per epoch: {len(val_loader)})")

# Verify sample batch shapes
sample_batch = next(iter(train_loader))
images = sample_batch["image"]
labels = sample_batch["label"]
print(f"Batch Image Shape: {images.shape}")   # Expected: (B, 4, 96, 96, 96)
print(f"Batch Label Shape: {labels.shape}")   # Expected: (B, 3, 96, 96, 96)

## 6. nnU-Net (DynUNet) Model, Loss, Optimizer, and Metrics Setup

Instantiate the MONAI `DynUNet` architecture with Instance Normalization and LeakyReLU.
We use `DiceCELoss(sigmoid=True)` and `AdamW` (or `SGD` with momentum) along with Cosine Annealing.

In [ ]:
# Instantiate DynUNet
model = build_nnunet_model(
    spatial_dims=config.spatial_dims,
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    filters=config.nnunet_filters,
    strides=config.nnunet_strides,
    kernel_size=config.nnunet_kernel_size,
    norm_name=config.nnunet_norm,
    act_name=config.nnunet_act,
    deep_supervision=config.nnunet_deep_supervision,
    res_block=config.nnunet_res_block,
    dropout=config.dropout if config.dropout > 0 else None
).to(config.device)

# Model summary
summary = get_model_summary(model)
print(f"Architecture: {summary['model_class']}")
print(f"Total Parameters: {summary['total_parameters']:,}")
print(f"Model Size: {summary['model_size_mb']} MB")

# Loss and metric
loss_fn = build_loss_fn(config.loss_type)
metric = BraTSMetric()

# Optimizer selection: AdamW or standard nnU-Net SGD with Nesterov momentum
if config.optimizer_type.lower() == "sgd":
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=1e-2,
        momentum=config.sgd_momentum,
        weight_decay=config.weight_decay,
        nesterov=config.sgd_nesterov
    )
else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay
    )

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.max_epochs,
    eta_min=1e-6
)

print(f"Optimizer: {optimizer.__class__.__name__}, Loss: {loss_fn.__class__.__name__}")

## 7. Pipeline Sanity Check

Verify 1 forward and backward pass with a sample batch to verify gradient flow and memory.

In [ ]:
model.train()
test_images = images[:1].to(config.device)
test_labels = labels[:1].to(config.device)

optimizer.zero_grad()
test_outputs = model(test_images)
test_loss = loss_fn(test_outputs, test_labels)
test_loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"Sanity check passed. Dummy batch loss: {test_loss.item():.4f}")
print(f"DynUNet output logits shape: {test_outputs.shape}")

## 8. Training and Validation Execution

Run training loop with sliding window inference validation.
The best checkpoint is automatically saved to `checkpoints/nnunet/best_metric_model.pth`.

In [ ]:
# Initialize Trainer
trainer = BraTSTrainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    config=config,
    lr_scheduler=lr_scheduler,
    metric=metric
)

# Number of epochs (e.g. 50-100 on GPU)
EPOCHS_TO_TRAIN = config.max_epochs
VAL_INTERVAL = config.val_interval

# Run training (run this on Google Colab with GPU)
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    max_epochs=EPOCHS_TO_TRAIN,
    val_interval=VAL_INTERVAL
)

## 9. Training Curves

Plot learning curves across epochs: training loss and validation Dice scores (TC, WT, ET, Mean Dice).

In [ ]:
curves_save_path = os.path.join(config.output_dir, "nnunet_training_curves.png")
fig_curves = plot_training_curves(
    history=trainer.history,
    save_path=curves_save_path
)
plt.show()

## 10. Qualitative Evaluation on Validation Case

Load the best checkpoint and evaluate on a full 3D volume using sliding window inference.

In [ ]:
from monai.inferers import sliding_window_inference

# Load best nnU-Net checkpoint if available
best_ckpt_path = os.path.join(config.checkpoint_dir, "best_metric_model.pth")
if os.path.exists(best_ckpt_path):
    load_checkpoint(best_ckpt_path, model, device=config.device)
    print(f"Loaded best checkpoint from: {best_ckpt_path}")
else:
    print(f"Checkpoint not found at {best_ckpt_path}; evaluating current model weights.")

# Run sliding window inference on 1 validation case
model.eval()
val_batch = next(iter(val_loader))
val_image = val_batch["image"].to(config.device)
val_label = val_batch["label"].to(config.device)

with torch.no_grad():
    val_pred = sliding_window_inference(
        inputs=val_image,
        roi_size=config.roi_size,
        sw_batch_size=config.sw_batch_size,
        predictor=model,
        overlap=config.overlap,
        mode="gaussian"
    )
    val_pred_mask = (torch.sigmoid(val_pred) > 0.5).float()

# Convert to numpy for visualization
img_np = val_image[0].cpu().numpy()
gt_np = val_label[0].cpu().numpy()
pred_np = val_pred_mask[0].cpu().numpy()

# Generate qualitative comparison
val_vis_path = os.path.join(config.output_dir, "nnunet_validation_prediction_sample.png")
fig_eval = visualize_predictions(
    image=img_np,
    label=gt_np,
    pred=pred_np,
    slice_idx=None,
    save_path=val_vis_path
)
plt.show()